# 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, Markdown
import ipywidgets as widgets
%load_ext autoreload
%autoreload 2


from Dataframes import dataframe_train, dafaframe_test, \
    transforma_df_em_csv, fazer_novo_df


# --------------------
# OPÇÕES DE MODELOS:
# --------------------
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LinearRegression


# 2. Extração de dados (48 Train e 19 Test)

In [2]:
from Dataframes import dafaframe_test, dataframe_train, \
    transforma_df_em_csv, fazer_novo_df

In [3]:
# dataframe_train()
# dafaframe_test()
# transforma_df_em_csv()
# fazer_novo_df()

In [4]:
print("Carregando bases de dados em CSV...")
df_treino_csv = pd.read_csv('Dados_Processados/treino_features.csv')
df_teste_csv = pd.read_csv('Dados_Processados/teste_features.csv')

# ========================================================
# O  DE SALVAMENTO: Recriando o Porta_ID caso ele não exista!
# Toda vez que o 'Ciclo' cai (ex: vai de 150 de volta para 1), ele soma +1 no ID.
# ========================================================
if 'Porta_ID' not in df_treino_csv.columns:
    print("Recriando a coluna Porta_ID no Treino...")
    df_treino_csv['Porta_ID'] = (df_treino_csv['Ciclo'] < df_treino_csv['Ciclo'].shift(1)).cumsum() + 1

if 'Porta_ID' not in df_teste_csv.columns:
    print("Recriando a coluna Porta_ID no Teste...")
    df_teste_csv['Porta_ID'] = (df_teste_csv['Ciclo'] < df_teste_csv['Ciclo'].shift(1)).cumsum() + 1

# 2. O truque: o groupby converte o DataFrame gigante de volta para a estrutura: [(1, df_porta1), ...]
portas_treino = list(df_treino_csv.groupby('Porta_ID'))
portas_teste = list(df_teste_csv.groupby('Porta_ID'))

print(f"✅ Sucesso! Encontradas {len(portas_treino)} portas de Treino e {len(portas_teste)} portas de Teste.")

Carregando bases de dados em CSV...
Recriando a coluna Porta_ID no Treino...
Recriando a coluna Porta_ID no Teste...
✅ Sucesso! Encontradas 43 portas de Treino e 19 portas de Teste.


# 3. Modelo

In [5]:
import os

total = 0
for i in range(1, 20):
    nome_pasta = f"Test/Test_{i}" 
    arquivos_csv = [arq for arq in os.listdir(nome_pasta) if arq.endswith('.csv')]        
    print(f"{i}: {len(arquivos_csv)}, ")
    total += len(arquivos_csv)
print(f"TOTAL: {total}")

1: 3593, 
2: 2358, 
3: 506, 
4: 2700, 
5: 6458, 
6: 2265, 
7: 1135, 
8: 1855, 
9: 1778, 
10: 3384, 
11: 3532, 
12: 2469, 
13: 2177, 
14: 2119, 
15: 2526, 
16: 1709, 
17: 2323, 
18: 2784, 
19: 1516, 
TOTAL: 47187


In [ ]:
X_train_list, y_train_list = [], []

# ====================================================================
# PASSO 1: PREPARAÇÃO DOS DADOS DE TREINO
# ====================================================================
# Lista unificada de colunas para remover do modelo (Evita o erro de Feature Missing!)
colunas_para_ignorar = ['Porta_ID', 'Ciclo', 'Ciclo_Relativo', 'RUL_Gabarito']

for porta_id, df_porta in portas_treino:
    df_features = df_porta.copy() 
    
    # 1. Calcula o RUL real (O alvo do modelo)
    if 'RUL_Gabarito' in df_features.columns:
        target = df_features['RUL_Gabarito']
    else:
        ciclo_da_falha = df_features['Ciclo'].max()
        target = ciclo_da_falha - df_features['Ciclo']
    
    # 2. Removemos a resposta e os relógios. O modelo foca apenas nos sensores!
    features = df_features.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_train_list.append(features)
    y_train_list.append(target)

# Junta todas as portas numa única matriz de aprendizagem
X_train_full = pd.concat(X_train_list, ignore_index=True).fillna(0)
y_train_full = np.concatenate(y_train_list)

print("A ajustar os hiperparâmetros (Cross-Validation) e treinando...")

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

modelo = LassoCV(alphas=np.logspace(-6, 1, 100), cv=10, max_iter=50000)
modelo.fit(X_train_full, y_train_full)

## -

In [ ]:
# ====================================================================
# PASSO 2: PREPARAÇÃO DOS DADOS DE TESTE
# ====================================================================
X_test_list = []

for porta_id, df_porta in portas_teste:
    df_features_teste = df_porta.copy() 
    
    # Removemos EXATAMENTE as mesmas colunas do treino
    features_teste = df_features_teste.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_test_list.append(features_teste)

# Junta todos os testes numa única matriz
X_test_full = pd.concat(X_test_list, ignore_index=True).fillna(0)

# ====================================================================
# PASSO 3: A PREDIÇÃO
# ====================================================================
print("Realizando as predições na base de Teste...")
RUL_predito = modelo.predict(X_test_full)
print("✅ Predições concluídas!")

import pandas as pd
import numpy as np

# ====================================================================
# PASSO 4: MONTANDO OS DADOS COM PREENCHIMENTO E GERANDO SUBMISSION
# ====================================================================
gabaritos_reais = {
    1: 3593, 2: 2358, 3: 506, 4: 2700, 5: 6458, 6: 2265, 7: 1135, 8: 1855, 9: 1778,\
    10: 3384, 11: 3532, 12: 2469, 13: 2177, 14: 2119, 15: 2526, 16: 1709, 17: 2323, 18: 2784, 19: 1516
}

# 1. Tratamos as predições do modelo PRIMEIRO (Evita que o RUL seja negativo ou decimal)
# Assim não corremos o risco de transformar os "0" do preenchimento em "1" depois.
RUL_tratado = np.clip(RUL_predito, 1, None).round().astype(int)

ids_teste = []
rul_alinhado = []
indice_corte = 0

# 2. Inserindo os zeros diretamente na formação das listas
for porta_id, df in portas_teste:
    tamanho_atual = len(df)
    linhas_faltantes = gabaritos_reais[porta_id] - tamanho_atual
    
    # Recorta exatamente as predições correspondentes a esta porta
    preds_reais_porta = RUL_tratado[indice_corte : indice_corte + tamanho_atual]
    
    # Se faltam linhas, adicionamos os IDs e os Zeros ANTES
    if linhas_faltantes > 0:
        ids_teste.extend([porta_id] * linhas_faltantes)
        rul_alinhado.extend([0] * linhas_faltantes)
        
    # Logo abaixo, adicionamos os IDs e os valores REAIS que o modelo previu
    ids_teste.extend([porta_id] * tamanho_atual)
    rul_alinhado.extend(preds_reais_porta)
    
    # Atualiza o ponto de corte para a próxima porta
    indice_corte += tamanho_atual

# 3. Sobrescrevemos a variável original conforme o seu pedido
RUL_predito = rul_alinhado

# 4. Criamos o df_resultados APENAS com a 1ª e 3ª colunas (ID e RUL)
df_resultados = pd.DataFrame({
    'Porta_ID': ids_teste,
    'RUL_Final': RUL_predito
})

print("✅ Preenchimento de zeros realizado direto no RUL_predito!")
print(f"Total de ciclos originais alinhados: {len(df_resultados)}")


A ajustar os hiperparâmetros (Cross-Validation) e treinando...
✅ Modelo treinado com sucesso!
Realizando as predições na base de Teste...
✅ Predições concluídas!
✅ Preenchimento de zeros realizado direto no RUL_predito!
Total de ciclos originais alinhados: 47187


#### Criação do submission:

In [7]:
# A quantidade de linhas finais exigidas por porta no Data Challenge
base = {
    1: 3593, 2: 2358, 3: 506, 4: 2700, 5: 6458, 6: 2265, 7: 1135, 8: 1855, 9: 1778,
    10: 3384, 11: 3532, 12: 2469, 13: 2177, 14: 2119, 15: 2526, 16: 1709, 17: 2323, 18: 2784, 19: 1516
}

RUL_final = []
ids_finais = []
indice_corte = 0

tamanho = 0

for porta_id, df in portas_teste:
    # 1. Identifica o tamanho original que a porta tem no seu RUL_predito atual
    tamanho_original = len(df)


    tamanho += base[porta_id]
    print(f"{porta_id}: {base[porta_id]}")

    
    # 2. Recorta apenas as predições correspondentes a essa porta
    preds_porta = RUL_predito[indice_corte : indice_corte + tamanho_original]
    
    # REGRA 1: Retirar o último valor (que é 0, fazendo terminar em 1)
    # Transforma em lista e fatia excluindo o último elemento [:-1]
    preds_porta_sem_ultimo = list(preds_porta)[:-1]
    
    # REGRA 2: Dobrar cada linha (ex: 4, 3, 2, 1 -> 4, 4, 3, 3, 2, 2, 1, 1)
    preds_dobradas = []
    for valor in preds_porta_sem_ultimo:
        preds_dobradas.extend([valor, valor])
        
    # REGRA 3: Preencher com "0" antes do primeiro valor até atingir o tamanho 'base'
    tamanho_atual_dobrado = len(preds_dobradas)
    linhas_faltantes = base[porta_id] - tamanho_atual_dobrado
    
    if linhas_faltantes > 0:
        # Cria a lista de zeros e soma (concatena) com a lista de predições dobradas
        preds_finais_porta = ([0] * linhas_faltantes) + preds_dobradas
    else:
        # Caso já tenha atingido o tamanho (proteção de segurança)
        preds_finais_porta = preds_dobradas
        
    # 4. Adiciona o resultado da porta nas listas mestres do submission
    RUL_final.extend(preds_finais_porta)
    ids_finais.extend([porta_id] * len(preds_finais_porta))
    
    # Atualiza o índice para cortar corretamente a predição da próxima porta
    indice_corte += tamanho_original

print(f"TOTAL: {tamanho}")


# ====================================================================
# GERANDO O ARQUIVO DE SUBMISSÃO FINAL
# ====================================================================
# Monta a tabela perfeitamente alinhada
df_submission = pd.DataFrame({
    'Porta_ID': ids_finais,
    'RUL_Final': RUL_final
})

# Salva no formato do Data Challenge
nome_arquivo = 'submission_final.csv'
df_submission.to_csv(nome_arquivo, sep=';', header=False, index=False)

print(f"✅ 'RUL_final' criado e '{nome_arquivo}' gerado com sucesso!")
print(f"TAMANHO SUBMISSION: {len(df_submission)}")

1: 3593
2: 2358
3: 506
4: 2700
5: 6458
6: 2265
7: 1135
8: 1855
9: 1778
10: 3384
11: 3532
12: 2469
13: 2177
14: 2119
15: 2526
16: 1709
17: 2323
18: 2784
19: 1516
TOTAL: 47187
✅ 'RUL_final' criado e 'submission_final.csv' gerado com sucesso!
TAMANHO SUBMISSION: 47187


In [10]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from IPython.display import display, HTML

# ========================================================
# FUNÇÃO DE NORMALIZAÇÃO MATEMÁTICA
# ========================================================
def norm_f(m, a=4443.76, b=1.53, c=4443.76, d=0):
    return (a / ((m**b) + c)) + d

arquivo_submissao = 'submission_final.csv'
arquivo_gabarito = 'gabarito_oficial.csv'

display(HTML("<h3>📊 Calculando o Score Oficial (Leitura Direta)</h3>"))

if not os.path.exists(arquivo_submissao) or not os.path.exists(arquivo_gabarito):
    print(f"❌ Erro: Arquivos '{arquivo_submissao}' ou '{arquivo_gabarito}' não encontrados.")
else:
    # ========================================================
    # 1. LEITURA DIRETA E CRIAÇÃO DO ERRO
    # ========================================================
    # Lemos os arquivos sem cabeçalho (header=None). 
    # Coluna 0 = Porta_ID | Coluna 1 = RUL
    df_sub = pd.read_csv(arquivo_submissao, sep=';', header=None)
    df_gab = pd.read_csv(arquivo_gabarito, sep=';', header=None)

    # Aplicando a sua lógica exata: pegamos a segunda coluna (índice 1)
    y_pred_total = df_sub[1]
    y_true_total = df_gab[1]
    
    # Criamos uma tabela simples apenas unindo essas colunas diretamente
    df_completo = pd.DataFrame({
        'Porta_ID': df_gab[0], # Pegamos o ID da primeira coluna
        'RUL_Pred': y_pred_total,
        'RUL_Real': y_true_total
    })

    # Como o desafio duplica as linhas (Opening/Closing), removemos as 
    # duplicatas sequenciais apenas para a contagem do tempo (T) ficar correta.
    df_unico = df_completo.drop_duplicates(keep='first').copy()
    
    # Recria o relógio (Ciclo_Atual)
    df_unico['Ciclo_Atual'] = df_unico.groupby('Porta_ID').cumcount() + 1

    # ========================================================
    # 2. CÁLCULO DAS MÉTRICAS POR PORTA
    # ========================================================
    alpha_peso = 2.0
    resultados_viagem = []

    for porta in df_unico['Porta_ID'].unique():
        
        df_porta = df_unico[df_unico['Porta_ID'] == porta].sort_values('Ciclo_Atual').copy()
        
        T = len(df_porta) 
        if T == 0: 
            continue
            
        y_true = df_porta['RUL_Real']
        y_pred = df_porta['RUL_Pred']
        
        # O ERRO DIRETO
        erro = y_true - y_pred
        
        # --- RMSE ---
        erro_quadrado = erro ** 2 
        mse = np.sum(erro_quadrado) / T
        rmse = np.sqrt(mse)

        # --- Precision ---
        erro_modulo = abs(erro) 
        fracao = erro_modulo / np.where(y_true == 0, 1e-9, y_true)
        
        fracoes_validas = fracao[fracao < 0.1]
        somatorio_prec = np.sum(fracoes_validas) if not fracoes_validas.empty else 0
        precicion = 100 * (somatorio_prec / T)

        # --- NORMALIZAÇÃO ---
        precicion_norm = norm_f(precicion)
        rmse_norm = norm_f(rmse)

        # --- PROGNOSTIC HORIZON (PH) ---
        t_eof = T  # t_eof é o total de linhas (fim da vida da porta)
        margem = T * 0.10  # A margem é exatamente 10% do total de linhas da porta
        
        # Calcula se a diferença direta (gabarito - submissão) está dentro da margem
        erro_absoluto = abs(df_porta['RUL_Real'] - df_porta['RUL_Pred'])
        
        dentro_dos_limites = erro_absoluto <= margem
        fora_dos_limites = ~dentro_dos_limites 
        
        # Mantendo exatamente a lógica que você validou:
        if fora_dos_limites.any():
            ultimo_erro = df_porta.loc[fora_dos_limites, 'Ciclo_Atual'].max()
            df_reta_final = df_porta[df_porta['Ciclo_Atual'] > ultimo_erro]
            
            # t_alpha se torna o índice da linha onde a predição ficou dentro da margem e não saiu mais
            t_alpha = t_eof if df_reta_final.empty else df_reta_final['Ciclo_Atual'].min()
        else:
            t_alpha = df_porta['Ciclo_Atual'].min()
                
        ph = (t_eof - t_alpha) / t_eof if t_eof > 0 else 0

        # --- SCORE FINAL ---
        score = (rmse_norm + precicion_norm + (alpha_peso * ph)) / (2 + alpha_peso)

        resultados_viagem.append({
            'Porta_ID': porta,
            'T_Pontos': T, 
            'Precision_Bruto': round(precicion, 2),
            'Precision_Norm': round(precicion_norm, 4),
            'RMSE_Bruto': round(rmse, 2),
            'RMSE_Norm': round(rmse_norm, 4),
            'PH': round(ph, 4),
            'Score': round(score, 4)
        })

    df_metricas = pd.DataFrame(resultados_viagem)

    # ========================================================
    # 3. TABELA INTERATIVA COM PLOTLY
    # ========================================================
    if not df_metricas.empty:
        fig = go.Figure(data=[go.Table(
            header=dict(
                values=[f"<b>{col}</b>" for col in df_metricas.columns], 
                fill_color='#2c3e50', 
                font=dict(color='white', size=13, family="Bookman Old Style, serif"),
                align='center',
                height=35
            ),
            cells=dict(
                values=[df_metricas[col] for col in df_metricas.columns],
                fill_color='#f5f6fa', 
                font=dict(color='black', size=12, family="Bookman Old Style, serif"),
                align='center',
                height=30
            )
        )])

        fig.update_layout(
            title=dict(text='<b>Score Oficial do Modelo</b>', x=0.5, font=dict(size=18, family="Bookman Old Style, serif")),
            margin=dict(l=20, r=20, t=50, b=20), 
            height=400 
        )

        fig.show()
        
        media_score = df_metricas['Score'].mean()
        display(HTML(f"""
        <div style="font-family: 'Bookman Old Style', serif; font-size: 22px; color: #27ae60; font-weight: bold; margin-top: 10px;">
            🏆 SCORE GLOBAL MÉDIO: {media_score:.4f}
        </div>
        """))

In [11]:
import plotly.graph_objects as go
import pandas as pd
import os

def comparar_submissao_vs_gabarito(arquivo_gabarito='gabarito_oficial.csv'):
    print("\nCarregando arquivos para auditoria: Previsão vs Realidade...")

    # Verifica se os arquivos existem antes de tentar abrir
    if not os.path.exists(arquivo_gabarito):
        print("❌ Erro: Certifique-se de que os arquivos 'submission.csv' e 'gabarito_oficial.csv' estão na pasta.")
        return

    
    df_gab = pd.read_csv(arquivo_gabarito, sep=';', header=None, names=['Porta_ID', 'RUL_Real'])

    # 2. Desfazemos as duplicatas (Opening/Closing) apenas para a visualização do gráfico
    df_final = df_final.drop_duplicates(keep='first').reset_index(drop=True)
    df_gab = df_gab.drop_duplicates(keep='first').reset_index(drop=True)

    # 3. Recria a coluna de Ciclos contando as linhas de cada porta (Eixo X)
    df_final['Ciclo'] = df_final.groupby('Porta_ID').cumcount() + 1
    df_gab['Ciclo'] = df_gab.groupby('Porta_ID').cumcount() + 1

    portas_testadas = df_final['Porta_ID'].unique()

    for porta in portas_testadas:
        df_p_sub = df_final[df_final['Porta_ID'] == porta]
        df_p_gab = df_gab[df_gab['Porta_ID'] == porta]

        # Inicia a figura (sem necessidade de eixos duplos agora!)
        fig = go.Figure()

        # -------------------------------------------------------------
        # LINHA 1: GABARITO (A realidade perfeita matemática)
        # -------------------------------------------------------------
        fig.add_trace(go.Scatter(
            x=df_p_gab['Ciclo'],
            y=df_p_gab['RUL_Real'],
            mode='lines',
            name='Gabarito (RUL Real)',
            line=dict(color='#2ca02c', width=3, dash='dash') # Verde Tracejado
        ))

        # -------------------------------------------------------------
        # LINHA 2: PREVISÃO (O seu modelo Ridge)
        # -------------------------------------------------------------
        fig.add_trace(go.Scatter(
            x=df_p_sub['Ciclo'],
            y=df_p_sub['RUL_Pred'],
            mode='lines',
            name='Previsão (RUL Modelo)',
            line=dict(color='#ff7f0e', width=3) # Laranja Sólido Vibrante
        ))

        # -------------------------------------------------------------
        # EMBELEZAMENTO E LAYOUT
        # -------------------------------------------------------------
        fig.update_layout(
            title=dict(text=f'<b>Auditoria Oficial - Porta Teste {porta}</b>', x=0.5, font=dict(size=18)),
            xaxis_title='<b>Tempo de Uso (Ciclos de Abertura)</b>',
            yaxis_title='<b>Ciclos Restantes (RUL)</b>',
            plot_bgcolor='white',          
            hovermode='x unified',         
            height=450,                    
            
            # Legenda no canto superior direito para não atrapalhar as linhas que caem para a esquerda
            legend=dict(
                yanchor="top", y=0.99,
                xanchor="right", x=0.99,
                bgcolor="rgba(255, 255, 255, 0.8)", 
                bordercolor="Black", borderwidth=1
            )
        )

        fig.update_xaxes(showgrid=True, gridcolor='#E5E5E5')
        fig.update_yaxes(showgrid=True, gridcolor='#E5E5E5')

        fig.show()



comparar_submissao_vs_gabarito()


Carregando arquivos para auditoria: Previsão vs Realidade...


UnboundLocalError: local variable 'df_final' referenced before assignment